## Imports

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


## Load data

In [ ]:
phd_students_df = pd.read_parquet("../data/phd_students.parquet")
phd_students_df.head(2)

In [ ]:
len(phd_students_df), len(phd_students_df["id_scopus_student"])

## Add new columns

In [ ]:
disciplines = ['AGRI', 'ARTS', 'BIOC', 'BUSI', 'CENG', 'CHEM', 'COMP', 'DECI', 'DENT', 'EART', 'ECON', 'ENER', 'ENGI', 'ENVI',
               'HEAL', 'IMMU', 'MATE', 'MATH', 'MEDI', 'MULT', 'NEUR', 'NURS', 'PHAR', 'PHYS', 'PSYC', 'SOCI', 'VETE']
len(disciplines), len(phd_students_df["discipline_student_scopus"].unique())

In [ ]:
index_MULT = disciplines.index("MULT")
index_MULT

### Nb pubs in MULT

In [ ]:
# Count the number of publications in MULT for each student
def count_mult(row):
    if row["areas_student"] is not None:
        areas = row["areas_student"].split("[")[1].split("]")[0].split(",")
        return float(areas[index_MULT])*int(row["num_pubs_student"])
    else:
        return 0
phd_students_df["nb_mult"] = phd_students_df.apply(count_mult, axis=1)

In [ ]:
len(phd_students_df[phd_students_df["nb_mult"] != 0])

In [ ]:
None in phd_students_df["name_supervisor1"].values, None in phd_students_df["name_supervisor2"].values, "" in phd_students_df["name_supervisor1"].values, "" in phd_students_df["name_supervisor2"].values, "nan" in phd_students_df["name_supervisor1"].values, "nan" in phd_students_df["name_supervisor2"].values

### Compare source id and issn/e-issn

In [ ]:
new_df = pd.read_excel("../data/journals_scopus.xlsx")
new_df.head(2)

In [ ]:
len(new_df), len(new_df["Sourcerecord ID"].unique())

In [ ]:
agri_df = pd.read_csv(r"C:\Users\sayfe\Desktop\PER\PER_temp\data\cleanScopus\clean_COMP.csv")
agri_df.head(2)

In [ ]:
len(agri_df), len(agri_df["source-id"].unique())

In [ ]:
agri_unclean_df = pd.read_csv(r"C:\Users\sayfe\Desktop\PER\PER_temp\data\scopus\COMP.csv")
agri_unclean_df.head(2)

In [ ]:
agri_unclean_df = agri_unclean_df[agri_unclean_df["subtype"]=="ar"]

In [ ]:
len(agri_unclean_df), len(agri_unclean_df["source-id"].unique())

In [ ]:
import pandas as pd

disciplines = ['AGRI', 'ARTS', 'BIOC', 'BUSI', 'CENG', 'CHEM', 'COMP', 'DECI', 'DENT', 'EART', 'ECON', 'ENER', 'ENGI', 'ENVI',
                'HEAL', 'IMMU', 'MATE', 'MATH', 'MEDI', 'MULT', 'NEUR', 'NURS', 'PHAR', 'PHYS', 'PSYC', 'SOCI', 'VETE']

len(disciplines)

In [ ]:
journals_df = pd.DataFrame(columns=["discipline", "source-id", "issn", "e-issn", "got_by_clean"])

for disc in disciplines :
    # Check path exists
    path = r"C:\Users\sayfe\Desktop\PER\PER_temp\data\cleanScopus\clean_" + disc + ".csv"
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        continue
    # Check path exists
    path = r"C:\Users\sayfe\Desktop\PER\PER_temp\data\scopus\\" + disc + ".csv"
    try:
        other_df = pd.read_csv(path)
    except FileNotFoundError:
        continue
    other_df = other_df[other_df["subtype"]=="ar"]
    other_nb_journals = len(other_df["source-id"].unique())
    # Add journals to dataframe
    df = df[["source-id", "prism:issn", "prism:eIssn"]]
    df["discipline"] = disc
    df["got_by_clean"] = True
    df = df.rename(columns={"prism:issn": "issn", "prism:eIssn": "e-issn"})
    journals_df = pd.concat([journals_df, df.drop_duplicates()])
    other_df = other_df[["source-id", "prism:issn", "prism:eIssn"]]
    other_df["discipline"] = disc
    other_df["got_by_clean"] = False
    other_df = other_df.rename(columns={"prism:issn": "issn", "prism:eIssn": "e-issn"})
    journals_df = pd.concat([journals_df, other_df.drop_duplicates()])

In [ ]:
journals_df.to_parquet("../data/sourceID_issn_compare.parquet", index=False)

In [ ]:
df.head(2)

## Work on sups

In [ ]:
def count_mult_sup(row, sup_index) :
    if row["areas_supervisor"+sup_index] is not None:
        areas = row["areas_supervisor"+sup_index].split("[")[1].split("]")[0].split(",")
        return float(areas[index_MULT])*int(row["num_pubs_supervisor"+sup_index])
    else:
        return 0

In [ ]:
for i in range(1, 3) :
    phd_students_df["nb_mult_sup"+str(i)] = phd_students_df.apply(count_mult_sup, axis=1, sup_index=str(i))

In [ ]:
def count_percentage_of_mult(row) :
    sum_mult = 0
    sum_pub = 0
    for i in range(1, 3) :
        sum_mult+= row["nb_mult_sup"+str(i)]
        sum_pub+= row["num_pubs_supervisor"+str(i)]
    return float(sum_mult)/float(sum_pub) if sum_pub != 0 else 0

In [ ]:
phd_students_df["percentage_of_mult_sup"] = phd_students_df.apply(count_percentage_of_mult, axis=1)

In [ ]:
phd_students_df["percentage_of_mult_student"] = phd_students_df["nb_mult"] / phd_students_df["num_pubs_student"]

## Fractions and Correlations

In [ ]:
import pandas as pd
phd_students_df = pd.read_parquet("../data/phd_students_mult.parquet", engine="pyarrow")

In [ ]:
phd_students_df.describe()

In [ ]:
len(phd_students_df[phd_students_df["percentage_of_mult_student"] > 0]), len(phd_students_df[phd_students_df["nb_mult"] > 0])

In [ ]:
len(phd_students_df[phd_students_df["nb_mult"] > 0]) / len(phd_students_df)

In [ ]:
# compute the correlation between the percentage of student publications in MULT and the percentage of supervisor publications in MULT
print("Corr %mult_student-%mult_sup", phd_students_df["percentage_of_mult_student"].corr(phd_students_df["percentage_of_mult_sup"]))
print("Corr %mult_student-%mult_sup spearman", phd_students_df["percentage_of_mult_student"].corr(phd_students_df["percentage_of_mult_sup"], method="spearman"))

In [ ]:
# Linear regression
x = phd_students_df[["percentage_of_mult_student"]]
y = phd_students_df["percentage_of_mult_sup"]
model = LinearRegression()
model.fit(x, y)
r2_score = model.score(x, y)
y_pred = model.predict(x)

In [ ]:
# compute the correlation between the number of student publications in MULT and the number of supervisor publications in MULT
print("Corr nb_mult-nb_mult_sup", phd_students_df["nb_mult"].corr(phd_students_df["nb_mult_sup1"]+phd_students_df["nb_mult_sup2"]))
print("Corr nb_mult-nb_mult_sup spearman", phd_students_df["nb_mult"].corr(phd_students_df["nb_mult_sup1"]+phd_students_df["nb_mult_sup2"], method="spearman"))

In [ ]:
# reduce same to student with supervisor distance above threshold 0.25
sample_df = phd_students_df[phd_students_df["distance_areas_supervisors"] > 0.25]
print(len(sample_df), " out of ", len(phd_students_df) , " → ", len(sample_df)/len(phd_students_df))
print("Corr %mult_student-%mult_sup", sample_df["percentage_of_mult_student"].corr(sample_df["percentage_of_mult_sup"]))
print("Corr nb_mult-nb_mult_sup spearman", sample_df["nb_mult"].corr(sample_df["nb_mult_sup1"]+sample_df["nb_mult_sup2"]))
# with spearman
print("Corr %mult_student-%mult_sup spearman", sample_df["percentage_of_mult_student"].corr(sample_df["percentage_of_mult_sup"], method="spearman"))
print("Corr nb_mult-nb_mult_sup spearman", sample_df["nb_mult"].corr(sample_df["nb_mult_sup1"]+sample_df["nb_mult_sup2"], method="spearman"))

In [ ]:
print(len(sample_df[sample_df["nb_mult"] > 0]), " out of ", len(sample_df) , " → ", len(sample_df[sample_df["nb_mult"] > 0])/len(sample_df))

In [ ]:
print("Corr %mult_student-num_pubs_student", sample_df["percentage_of_mult_student"].corr(sample_df["num_pubs_student"]))

In [ ]:
print("Corr %mult_student-citations_student", sample_df["percentage_of_mult_student"].corr(sample_df["citations_student"]))

In [ ]:
print("Corr %mult_sup-num_pubs_student", phd_students_df["percentage_of_mult_sup"].corr(phd_students_df["num_pubs_student"]))

# Temporary code

In [ ]:
import pandas as pd

In [ ]:
journals_info = pd.read_parquet("../data/journals_info.parquet")
journals_info.head(2)

In [ ]:
journals_scopus = pd.read_excel("../data/journals_scopus.xlsx")
journals_scopus.head(2)

In [ ]:
# add new journal_id column to journals_scopus : first available issn or e-issn
def add_id(row):
    if pd.notna(row["ISSN"]):
        return row["ISSN"]
    elif pd.notna(row["EISSN"]):
        return row["EISSN"]
    else:
        return None

journals_scopus["journal_id"] = journals_scopus.apply(add_id, axis=1)

In [ ]:
# set type string for id
journals_info["journal_id"] = journals_info["journal_id"].astype(str)
journals_scopus["journal_id"] = journals_scopus["journal_id"].astype(str)

# merge the two dataframes on the "source-id" column
merged_df = journals_info.join(journals_scopus.set_index("journal_id"), on="journal_id")

In [ ]:
merged_df.head(2)

In [ ]:
# to parquet
merged_df.to_parquet("../data/journals_info_scopus.parquet", index=False)

In [ ]:
journals_umap_selection = pd.read_csv("../data/journals_umap1_lt_3.csv")
journals_umap_selection.head(2)

In [ ]:
# set of journal_ids
journals_umap_selection["journal_id"] = journals_umap_selection.apply(add_id, axis=1)

In [ ]:
merged_df["is_weird_point"] = merged_df.apply(lambda row: row["journal_id"] in journals_umap_selection["journal_id"].values, axis=1)